In [ ]:
# ============================================================
# BLOCK 0: Environment setup
# ============================================================
!pip install transformers datasets evaluate seqeval -q

import os, re, json
import pandas as pd
import numpy as np
from datasets import Dataset
import evaluate
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer,
    Trainer, DataCollatorForTokenClassification
)

seqeval = evaluate.load("seqeval")

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/MABSA_Shared_Project/data/master_dataset.parquet")
print("df loaded:", df.shape)

In [ ]:
# ============================================================
# BLOCK 1: Helper functions (with punctuation fix + safe tokenization)
# ============================================================

def make_bio_tags(sentence, aspect, category, sentiment, task="tasd"):
    tokens = sentence.split()
    tags = ["O"] * len(tokens)
    if pd.isna(aspect) or str(aspect).strip() in ("", "NULL"):
        return tokens, tags

    aspect_tokens = aspect.split()
    n = len(aspect_tokens)

    # Strip leading/trailing punctuation for matching only (fixes German/Arabic/Spanish
    # mismatch issue found during RQ3 — includes Arabic comma ، and parens)
    def strip_punct(w):
        return re.sub(r'^[.,!?;:"\'()\[\]،]+|[.,!?;:"\'()\[\]،]+$', '', w)

    clean_tokens = [strip_punct(t) for t in tokens]
    clean_aspect_tokens = [strip_punct(t) for t in aspect_tokens]

    for i in range(len(clean_tokens) - n + 1):
        if clean_tokens[i:i+n] == clean_aspect_tokens:
            if task == "tasd":
                tags[i] = f"B-{category}-{sentiment}"
                for j in range(1, n):
                    tags[i+j] = f"I-{category}-{sentiment}"
            else:
                tags[i] = f"B-{sentiment}"
                for j in range(1, n):
                    tags[i+j] = f"I-{sentiment}"
            break
    return tokens, tags

def build_tagged_df(source_df, task="tasd"):
    records = []
    for _, row in source_df.iterrows():
        tokens, tags = make_bio_tags(row.sentence, row.aspect, row.category, row.sentiment, task=task)
        records.append({"tokens": tokens, "tags": tags, "split": row.split})
    return pd.DataFrame(records)

def make_tokenize_fn_safe(label2id_local, tok):
    """Unseen categories map to -100 (excluded from scoring) instead of crashing."""
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tok(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids, prev = [], None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != prev:
                    label_ids.append(label2id_local.get(tags[word_idx], -100))
                else:
                    label_ids.append(-100)
                prev = word_idx
            all_labels.append(label_ids)
        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs
    return tokenize_and_align_labels

def make_compute_metrics(label_list_local):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        true_labels = [[label_list_local[l] for l in label if l != -100] for label in p.label_ids]
        true_preds = [[label_list_local[pp] for (pp, l) in zip(pred, label) if l != -100]
                      for pred, label in zip(preds, p.label_ids)]
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}
    return compute_metrics

def spot_check_domain_lang(test_df, label_name, n=50):
    tagged = build_tagged_df(test_df, task="tasd").reset_index(drop=True)
    non_null = test_df[test_df.aspect.notna() & (test_df.aspect.astype(str).str.strip() != "NULL")]
    sample_idx = non_null.index[:n]
    sample_check = tagged.loc[sample_idx]
    unmatched = sum(1 for t in sample_check["tags"] if all(x == "O" for x in t))
    print(f"{label_name}: {unmatched}/{len(sample_idx)} unmatched")
    return unmatched, tagged

In [ ]:
# ============================================================
# BLOCK 2: Load existing RQ1 checkpoint (English, restaurant, TASD)
# ============================================================
CKPT = "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_tasd_seed42_final"

model = AutoModelForTokenClassification.from_pretrained(CKPT)
tokenizer = AutoTokenizer.from_pretrained(CKPT)
with open(os.path.join(CKPT, "label_map.json")) as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}
label_list = [id2label[i] for i in range(len(id2label))]
print(f"Checkpoint loaded. Total labels: {len(label2id)}")

Loading weights:   0%|          | 0/199 [00:04<?, ?it/s]

Checkpoint loaded. Total labels: 79


In [ ]:
# ============================================================
# BLOCK 3: Build test sets for RQ2 languages (restaurant domain)
# ============================================================
language_iso_map = {
    "swahili": "sw", "arabic": "ar", "hindi": "hi",
    "thai": "th", "croatian": "hr", "vietnamese": "vi"
}
languages = list(language_iso_map.keys())

# Row counts first
for lang_name, iso in language_iso_map.items():
    n = len(df[(df.language == iso) & (df.split == "test") & (df.domain == "restaurant")])
    print(f"{lang_name} ({iso}): {n} test rows")

swahili (sw): 788 test rows
arabic (ar): 797 test rows
hindi (hi): 799 test rows
thai (th): 793 test rows
croatian (hr): 793 test rows
vietnamese (vi): 796 test rows


In [ ]:
test_sets = {}
for lang_name, iso in language_iso_map.items():
    lang_test_df = df[(df.language == iso) & (df.split == "test") & (df.domain == "restaurant")].reset_index(drop=True)
    unmatched, lang_tagged = spot_check_domain_lang(lang_test_df, lang_name)
    test_sets[lang_name] = Dataset.from_pandas(lang_tagged[["tokens", "tags"]])

print("\nTest sets built:", list(test_sets.keys()))

swahili: 0/50 unmatched
arabic: 3/50 unmatched
hindi: 0/50 unmatched
thai: 3/50 unmatched
croatian: 0/50 unmatched
vietnamese: 0/50 unmatched

Test sets built: ['swahili', 'arabic', 'hindi', 'thai', 'croatian', 'vietnamese']


In [ ]:
# ============================================================
# BLOCK 4: Feature 1 — Zero-shot eval across all 6 languages
# ============================================================
tok_fn = make_tokenize_fn_safe(label2id, tokenizer)
collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(model=model, processing_class=tokenizer, data_collator=collator,
                   compute_metrics=make_compute_metrics(label_list))

rq2_feature1_results = {}
for lang in languages:
    tokenized_test = test_sets[lang].map(tok_fn, batched=True)
    preds = trainer.predict(tokenized_test)
    rq2_feature1_results[lang] = make_compute_metrics(label_list)(preds)["f1"]
    print(f"{lang}: F1 = {rq2_feature1_results[lang]:.4f}")

print("\n=== RQ2 Feature 1 results ===")
print(rq2_feature1_results)

out_path = "/content/drive/MyDrive/MABSA_Shared_Project/results/rq2_feature1_zeroshot.json"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w") as f:
    json.dump(rq2_feature1_results, f, indent=2)
print(f"Saved to {out_path}")

Map:   0%|          | 0/788 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


swahili: F1 = 0.2183


Map:   0%|          | 0/797 [00:00<?, ? examples/s]

arabic: F1 = 0.3156


...

thai: F1 = 0.1919


Map:   0%|          | 0/793 [00:00<?, ? examples/s]

croatian: F1 = 0.3557


Map:   0%|          | 0/796 [00:00<?, ? examples/s]

vietnamese: F1 = 0.1624

=== RQ2 Feature 1 results ===
{'swahili': np.float64(0.21832358674463936), 'arabic': np.float64(0.3155818540433925), 'hindi': np.float64(0.3046357615894039), 'thai': np.float64(0.19189189189189193), 'croatian': np.float64(0.3557168784029038), 'vietnamese': np.float64(0.16239316239316237)}
Saved to /content/drive/MyDrive/MABSA_Shared_Project/results/rq2_feature1_zeroshot.json


In [ ]:
# ============================================================
# BLOCK 5: Feature 2 — Language family analysis
# ============================================================
language_family = {
    "swahili": "Bantu",
    "arabic": "Semitic",
    "hindi": "Indo-Aryan",
    "thai": "Kra-Dai",
    "croatian": "Slavic",
    "vietnamese": "Austroasiatic",
}

language_indo_european = {
    "swahili": False, "arabic": False, "hindi": True,
    "thai": False, "croatian": True, "vietnamese": False,
}

for lang in languages:
    fam = language_family[lang]
    f1 = rq2_feature1_results[lang]
    ie = "Indo-European" if language_indo_european[lang] else "Non-Indo-European"
    print(f"{lang:12s} | Family: {fam:15s} | {ie:20s} | F1: {f1:.4f}")

# Ranked view
ranked = sorted(languages, key=lambda l: rq2_feature1_results[l], reverse=True)
print("\nRanked best to worst:")
for lang in ranked:
    print(f"  {lang}: F1={rq2_feature1_results[lang]:.4f} ({language_family[lang]})")

swahili      | Family: Bantu           | Non-Indo-European    | F1: 0.2183
arabic       | Family: Semitic         | Non-Indo-European    | F1: 0.3156
hindi        | Family: Indo-Aryan      | Indo-European        | F1: 0.3046
thai         | Family: Kra-Dai         | Non-Indo-European    | F1: 0.1919
croatian     | Family: Slavic          | Indo-European        | F1: 0.3557
vietnamese   | Family: Austroasiatic   | Non-Indo-European    | F1: 0.1624

Ranked best to worst:
  croatian: F1=0.3557 (Slavic)
  arabic: F1=0.3156 (Semitic)
  hindi: F1=0.3046 (Indo-Aryan)
  swahili: F1=0.2183 (Bantu)
  thai: F1=0.1919 (Kra-Dai)
  vietnamese: F1=0.1624 (Austroasiatic)
